<a href="https://colab.research.google.com/github/DeepthiManthapuram/RAG/blob/main/FAISS_vectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
chunks = [

    "Employees receive 12 casual leaves annually.",

    "Employees receive 15 sick leaves annually.",

    "Employees may work from home twice per week.",

    "Travel expenses are reimbursed within 30 days.",

    "All employees are covered under company medical insurance."

]

print("Total Chunks:", len(chunks))


Total Chunks: 5


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
"all-MiniLM-L6-v2"

)
print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


Generate Embeddings

In [3]:
embeddings=model.encode(chunks)
print("Embedding SHape:")
print(embeddings. shape)


Embedding SHape:
(5, 384)


create FAISS index

In [4]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 97.2 MB/s eta 0:00:00


In [5]:
import faiss

dimension=embeddings.shape[1]

index=faiss.IndexFlatL2(dimension)

print("FAISS Index Created")

FAISS Index Created


Add Embeddings to FAISS

In [6]:
import numpy as np
index.add(
np.array(embeddings,
dtype=np.float32)
)
print("vectors Stored:",index.ntotal)

vectors Stored: 5


User Query

In [7]:
query="How many casual leaves are allowd?"
print(query)

How many casual leaves are allowd?


Convert Query into Embeddings

In [8]:
query_embedding=model.encode(
[query]
)

Search FAISS

In [9]:
k=3
distances, indices=index.search(
np.array(
query_embedding,
dtype=np.float32

),
  k
)

Display Results

In [10]:
print("Maching Chunks\n")

for rank,idx in enumerate(indices[0]):
  print(f'Rank {rank+1}:')
  print(chunks[idx])
  print("Distance:",distances[0][rank])
  print("-"*50)

Maching Chunks

Rank 1:
Employees receive 12 casual leaves annually.
Distance: 0.7992542
--------------------------------------------------
Rank 2:
Employees receive 15 sick leaves annually.
Distance: 1.2528427
--------------------------------------------------
Rank 3:
Employees may work from home twice per week.
Distance: 1.7353495
--------------------------------------------------


Intelligent Employee Policy Assistant using RAG
Problem Statement
A multinational company maintains thousands of employee-related documents, including HR policies, leave regulations, travel reimbursement guidelines, medical insurance policies, and work-from-home procedures.
Employees frequently ask questions about company policies. Searching through multiple PDF documents manually is time-consuming and inefficient.
Your task is to build a Retrieval-Augmented Generation (RAG) system that can answer employee questions accurately by retrieving relevant information from company policy documents and generating human-readable responses using a Large Language Model.
Dataset
Create and use the following PDFs:
Employee Handbook.pdf
Leave Policy.pdf
Travel Policy.pdf
Work From Home Policy.pdf
Medical Insurance Policy.pdf
You may create sample content or use real HR policy documents.
Task 1: Document Loading
Objective
Load all company policy PDFs.
Requirements
Read multiple PDF files.
Extract text content.
Display document statistics.  
Deliverables
Display:
File Name
Number of Pages
Total Characters
Total Words
Task 2: Document Chunking
Objective
Split large policy documents into smaller chunks.
Requirements
Implement:
Fixed Size Chunking
Recursive Chunking
Use:
Chunk Size = 500
Chunk Overlap = 100
Deliverables
Display:
Chunk Number
Chunk Content
Chunk Length
Analysis
Explain:
Why chunking is necessary in RAG.

In [11]:
!pip install reportlab pypdf langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 34.6 MB/s eta 0:00:00


In [12]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# Define sample content for each policy file
policy_data = {
    "Employee Handbook.pdf": (
        "Welcome to the Company Employee Handbook. Our organization is dedicated to fostering a diverse, "
        "inclusive, and productive working environment. All employees are expected to maintain the highest "
        "standards of professional integrity, ethical conduct, and mutual respect. Compliance with company "
        "security, data privacy, and intellectual property agreements is strictly mandatory for all roles. "
        "Detailed performance reviews are conducted annually to align career progression with company milestones."
    ),
    "Leave Policy.pdf": (
        "Standard Leave Policy Guidelines. Employees are granted a total of 12 casual leaves annually, which "
        "accrue on a monthly basis. Additionally, 15 sick leaves are allocated per calendar year for medical "
        "emergencies. Maternity leave extends up to 26 weeks of paid time off, while paternity leave offers "
        "2 weeks of paid time off. All planned leaves must be submitted through the HR portal and approved by "
        "the immediate manager at least 5 business days in advance."
    ),
    "Travel Policy.pdf": (
        "Corporate Travel Reimbursement Policy. Travel expenses incurred for official company business will be "
        "reimbursed within 30 days of submitting a valid expense claim. Economy class is mandatory for all flights "
        "under 6 hours. Accommodation limits are capped at $150 per night for domestic travel and $250 per night "
        "for international travel. Original receipts and digital invoices must accompany all claims submitted via "
        "the corporate expense portal."
    ),
    "Work From Home Policy.pdf": (
        "Flexible Work Arrangements and Work From Home Policy. Employees may work from home twice per week, "
        "subject to team alignment and operational requirements. Core operating hours are 10:00 AM to 4:00 PM. "
        "The company provides a one-time remote workspace stipend of $300 to purchase ergonomic chairs, monitors, "
        "or high-speed internet routers. Employees must ensure a secure, distraction-free environment and reliable "
        "network access during remote work blocks."
    ),
    "Medical Insurance Policy.pdf": (
        "Company Medical Insurance Plan Details. All full-time employees are covered under the company comprehensive "
        "medical insurance policy starting from their first day of employment. The policy offers a maximum annual "
        "coverage limit of $50,000 per family unit. Outpatient care, prescription drugs, dental consultations, and "
        "vision checks are covered up to 80 percent of total costs. Hospitalization pre-authorization must be filed "
        "24 hours prior to elective admissions."
    )
}

def create_pdf(filename, content):
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()

    # Custom heading and body styles
    title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontSize=18, spaceAfter=12)
    body_style = ParagraphStyle('DocBody', parent=styles['Normal'], fontSize=11, leading=16)

    story = [
        Paragraph(filename.replace(".pdf", ""), title_style),
        Spacer(1, 12),
        # Repeating text to ensure files span realistic lengths/character counts
        Paragraph(content * 6, body_style)
    ]
    doc.build(story)

# Generate all 5 files
print("Generating PDF files...")
for filename, text in policy_data.items():
    create_pdf(filename, text)
print("All 5 policy PDFs generated successfully!")

Generating PDF files...
All 5 policy PDFs generated successfully!


In [13]:
import pypdf

pdf_files = list(policy_data.keys())
documents_text = {}

print("=" * 70)
print(f"{'File Name':<30} | {'Pages':<6} | {'Characters':<10} | {'Words':<8}")
print("=" * 70)

for file_name in pdf_files:
    try:
        reader = pypdf.PdfReader(file_name)
        num_pages = len(reader.pages)

        # Extract content page by page
        full_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"

        documents_text[file_name] = full_text

        # Compute structural statistics
        total_chars = len(full_text)
        total_words = len(full_text.split())

        print(f"{file_name:<30} | {num_pages:<6} | {total_chars:<10} | {total_words:<8}")

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

print("=" * 70)

File Name                      | Pages  | Characters | Words   
Employee Handbook.pdf          | 1      | 2984       | 387     
Leave Policy.pdf               | 1      | 2763       | 453     
Travel Policy.pdf              | 1      | 2692       | 393     
Work From Home Policy.pdf      | 1      | 2742       | 395     
Medical Insurance Policy.pdf   | 1      | 2811       | 394     


In [14]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Target file for chunking demonstration
sample_file = "Leave Policy.pdf"
sample_content = documents_text[sample_file]

# -------------------------------------------------------------
# 1. Fixed-Size Chunking Implementation
# -------------------------------------------------------------
print(f"\n=== FIXED SIZE CHUNKING DEMO ({sample_file}) ===")
fixed_splitter = CharacterTextSplitter(
    separator="", # Strict fixed character calculation
    chunk_size=500,
    chunk_overlap=100
)
fixed_chunks = fixed_splitter.split_text(sample_content)

for idx, chunk in enumerate(fixed_chunks):
    print(f"Chunk Number: {idx + 1}")
    print(f"Chunk Length: {len(chunk)} characters")
    print(f"Chunk Content:\n{chunk.strip()}")
    print("-" * 50)


# -------------------------------------------------------------
# 2. Recursive Chunking Implementation
# -------------------------------------------------------------
print(f"\n=== RECURSIVE CHUNKING DEMO ({sample_file}) ===")
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""] # Hierarchical priority checks
)
recursive_chunks = recursive_splitter.split_text(sample_content)

for idx, chunk in enumerate(recursive_chunks):
    print(f"Chunk Number: {idx + 1}")
    print(f"Chunk Length: {len(chunk)} characters")
    print(f"Chunk Content:\n{chunk.strip()}")
    print("-" * 50)


=== FIXED SIZE CHUNKING DEMO (Leave Policy.pdf) ===
Chunk Number: 1
Chunk Length: 500 characters
Chunk Content:
Leave Policy
Standard Leave Policy Guidelines. Employees are granted a total of 12 casual leaves
annually, which accrue on a monthly basis. Additionally, 15 sick leaves are allocated per
calendar year for medical emergencies. Maternity leave extends up to 26 weeks of paid time
off, while paternity leave offers 2 weeks of paid time off. All planned leaves must be submitted
through the HR portal and approved by the immediate manager at least 5 business days in
advance.Standard Leave Policy Guideli
--------------------------------------------------
Chunk Number: 2
Chunk Length: 499 characters
Chunk Content:
approved by the immediate manager at least 5 business days in
advance.Standard Leave Policy Guidelines. Employees are granted a total of 12 casual
leaves annually, which accrue on a monthly basis. Additionally, 15 sick leaves are allocated
per calendar year for medical emerg

In [15]:
import os
import pypdf

# List of required policy PDFs
pdf_files = [
    "Employee Handbook.pdf",
    "Leave Policy.pdf",
    "Travel Policy.pdf",
    "Work From Home Policy.pdf",
    "Medical Insurance Policy.pdf"
]

# Dictionary to hold extracted text for Task 2
documents_text = {}

print("="*60)
print(f"{'File Name':<30} | {'Pages':<6} | {'Characters':<10} | {'Words':<8}")
print("="*60)

for file_name in pdf_files:
    if not os.path.exists(file_name):
        # Fallback: Create a dummy PDF if the file doesn't exist so the code runs seamlessly
        from pypdf import PdfWriter
        writer = PdfWriter()
        writer.add_blank_page(width=612, height=792)
        with open(file_name, "wb") as f:
            writer.write(f)

    try:
        reader = pypdf.PdfReader(file_name)
        num_pages = len(reader.pages)

        # Extract full text
        full_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"

        # If it was a dummy/empty PDF, add some sample text for demonstration
        if not full_text.strip():
            full_text = f"This is mock content for {file_name}. " * 100

        documents_text[file_name] = full_text

        # Calculate statistics
        total_chars = len(full_text)
        total_words = len(full_text.split())

        print(f"{file_name:<30} | {num_pages:<6} | {total_chars:<10} | {total_words:<8}")

    except Exception as e:
        print(f"Error processing {file_name}: {e}")

print("="*60)

File Name                      | Pages  | Characters | Words   
Employee Handbook.pdf          | 1      | 2984       | 387     
Leave Policy.pdf               | 1      | 2763       | 453     
Travel Policy.pdf              | 1      | 2692       | 393     
Work From Home Policy.pdf      | 1      | 2742       | 395     
Medical Insurance Policy.pdf   | 1      | 2811       | 394     


In [16]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Choose a sample loaded document to demonstrate chunking outputs
sample_file = "Leave Policy.pdf"
sample_content = documents_text.get(sample_file, "Sample policy text goes here.")

print(f"--- Demonstrating Chunking on: {sample_file} ---\n")

# -------------------------------------------------------------
# 1. Fixed-Size Chunking
# -------------------------------------------------------------
print("### IMPLEMENTING FIXED SIZE CHUNKING ###")
fixed_splitter = CharacterTextSplitter(
    separator="",          # Fixed-size strictly by character count
    chunk_size=500,
    chunk_overlap=100
)
fixed_chunks = fixed_splitter.split_text(sample_content)

for i, chunk in enumerate(fixed_chunks[:3]): # Displaying first 3 chunks for brevity
    print(f"Chunk Number: {i + 1}")
    print(f"Chunk Length: {len(chunk)} characters")
    print(f"Chunk Content:\n{chunk.strip()}")
    print("-" * 50)


# -------------------------------------------------------------
# 2. Recursive Chunking
# -------------------------------------------------------------
print("\n### IMPLEMENTING RECURSIVE CHUNKING ###")
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""] # Order of structural priority
)
recursive_chunks = recursive_splitter.split_text(sample_content)

for i, chunk in enumerate(recursive_chunks[:3]): # Displaying first 3 chunks for brevity
    print(f"Chunk Number: {i + 1}")
    print(f"Chunk Length: {len(chunk)} characters")
    print(f"Chunk Content:\n{chunk.strip()}")
    print("-" * 50)

--- Demonstrating Chunking on: Leave Policy.pdf ---

### IMPLEMENTING FIXED SIZE CHUNKING ###
Chunk Number: 1
Chunk Length: 500 characters
Chunk Content:
Leave Policy
Standard Leave Policy Guidelines. Employees are granted a total of 12 casual leaves
annually, which accrue on a monthly basis. Additionally, 15 sick leaves are allocated per
calendar year for medical emergencies. Maternity leave extends up to 26 weeks of paid time
off, while paternity leave offers 2 weeks of paid time off. All planned leaves must be submitted
through the HR portal and approved by the immediate manager at least 5 business days in
advance.Standard Leave Policy Guideli
--------------------------------------------------
Chunk Number: 2
Chunk Length: 499 characters
Chunk Content:
approved by the immediate manager at least 5 business days in
advance.Standard Leave Policy Guidelines. Employees are granted a total of 12 casual
leaves annually, which accrue on a monthly basis. Additionally, 15 sick leaves are allo

In [17]:
from sentence_transformers import SentenceTransformer

# 1. Load the requested embedding model
print("Loading all-MiniLM-L6-v2 model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully.\n")

# Use the recursive chunks generated from Leave Policy.pdf in Task 2
chunks_to_embed = recursive_chunks

# 2. Generate vector embeddings
embeddings = embedding_model.encode(chunks_to_embed)

# 3. Deliverables: Display metrics for the first 2 chunks as a sample
for idx, (chunk, embedding) in enumerate(zip(chunks_to_embed[:2], embeddings[:2])):
    print(f"=== Chunk {idx + 1} ===")
    print(f"Text Content snippet: {chunk.strip()[:120]}...")
    print(f"Embedding Shape: {embedding.shape}")
    # Displaying the first 5 dimensions as sample values
    print(f"Sample Embedding Values (First 5 dimensions): {embedding[:5]}\n")
    print("-" * 60)

Loading all-MiniLM-L6-v2 model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully.

=== Chunk 1 ===
Text Content snippet: Leave Policy
Standard Leave Policy Guidelines. Employees are granted a total of 12 casual leaves
annually, which accrue ...
Embedding Shape: (384,)
Sample Embedding Values (First 5 dimensions): [ 0.0483988   0.00030957 -0.03122957  0.04283947  0.13820155]

------------------------------------------------------------
=== Chunk 2 ===
Text Content snippet: through the HR portal and approved by the immediate manager at least 5 business days in
advance.Standard Leave Policy Gu...
Embedding Shape: (384,)
Sample Embedding Values (First 5 dimensions): [ 0.04943411 -0.00205664 -0.02885693  0.04575858  0.13376208]

------------------------------------------------------------


In [18]:
import faiss
import numpy as np

# 1. Gather structural dimensions
embedding_dimension = embeddings.shape[1]  # Expected to be 384 for all-MiniLM-L6-v2
total_chunks_processed = len(chunks_to_embed)

# 2. Instantiate a flat L2 distance FAISS Index
# IndexFlatL2 performs a brute-force L2 (Euclidean) distance search for exact precision
faiss_index = faiss.IndexFlatL2(embedding_dimension)

# 3. Cast embeddings matrix to float32 (required by FAISS C++ backend) and insert
embeddings_matrix = np.array(embeddings, dtype=np.float32)
faiss_index.add(embeddings_matrix)

# 4. Deliverables
print("=" * 50)
print(f"{'FAISS Vector Database Statistics':^50}")
print("=" * 50)
print(f"Number of Input Chunks:      {total_chunks_processed}")
print(f"Number of Stored Vectors:    {faiss_index.ntotal}")
print(f"Embedding Dimension:         {embedding_dimension}")
print("=" * 50)

         FAISS Vector Database Statistics         
Number of Input Chunks:      8
Number of Stored Vectors:    8
Embedding Dimension:         384


In [19]:
import numpy as np
queries=[
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]
print("=" * 80)
print(f"{'SEMANTIC RETRIEVAL SYSTEM EVALUATION':^80}")
print("=" * 80)

# 2. Execute retrieval for each query
for query in queries:
    print(f"\n📢 Employee Query: '{query}'")
    print("-" * 80)

    # Encode user query to the same 384-dimensional space
    query_vector = embedding_model.encode([query])
    query_vector_np = np.array(query_vector, dtype=np.float32)

    # Search FAISS index for the Top 3 nearest neighbors (k=3)
    # L2 distance means: Lower Score = Closer Distance = Higher Similarity
    k = 3
    distances, indices = faiss_index.search(query_vector_np, k)

    # 3. Deliverables: Display Top 3 Matches
    for rank in range(k):
        chunk_idx = indices[0][rank]
        distance_score = distances[0][rank]

        # Guard against index out of bounds if testing with fewer initial chunks
        if chunk_idx < len(chunks_to_embed):
            retrieved_text = chunks_to_embed[chunk_idx].strip()
            # Truncate slightly for clean notebook display if text is long
            display_text = retrieved_text if len(retrieved_text) <= 250 else retrieved_text[:245] + "..."

            print(f"🥇 Match Rank {rank + 1}")
            print(f"🔹 FAISS Distance (L2 Score): {distance_score:.4f}")
            print(f"📝 Retrieved Chunk Content:\n{display_text}")
            print("." * 40)
        else:
            print(f"Rank {rank + 1}: No valid chunk found matching index.")

    print("=" * 80)

                      SEMANTIC RETRIEVAL SYSTEM EVALUATION                      

📢 Employee Query: 'How many casual leaves are available?'
--------------------------------------------------------------------------------
🥇 Match Rank 1
🔹 FAISS Distance (L2 Score): 1.1931
📝 Retrieved Chunk Content:
of 12 casual leaves annually, which accrue on a monthly basis. Additionally, 15 sick leaves
are allocated per calendar year for medical emergencies. Maternity leave extends up to 26
weeks of paid time off, while paternity leave offers 2 weeks of...
........................................
🥇 Match Rank 2
🔹 FAISS Distance (L2 Score): 1.2380
📝 Retrieved Chunk Content:
must be submitted through the HR portal and approved by the immediate manager at least 5
business days in advance.Standard Leave Policy Guidelines. Employees are granted a total
of 12 casual leaves annually, which accrue on a monthly basis. Addi...
........................................
🥇 Match Rank 3
🔹 FAISS Distance (L2 Score):

In [20]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Initialize the open, un-gated Phi-2 model
model_id = "microsoft/phi-2"
print(f"Loading LLM: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("LLM loaded successfully!\n")

# 2. Setup the Sample Query & Context
user_question = "How many casual leaves are available?"
retrieved_chunks = (
    "Standard Leave Policy Guidelines. Employees are granted a total of 12 casual leaves annually, "
    "which accrue on a monthly basis. Additionally, 15 sick leaves are allocated per calendar year for "
    "medical emergencies. Maternity leave extends up to 26 weeks of paid time off."
)

# 3. Enhanced Prompt Format (Prevents the model from creating math/exam exercises)
prompt = f"""Instruction: You are an automated HR policy assistant. Provide a single, direct, factual answer to the question using only the context given below. Do not generate alternative scenarios or continue typing after your answer.

Context:
{retrieved_chunks}

Question:
{user_question}

Answer:"""

# 4. Tokenize and Generate
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating Answer...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,       # Lowered token budget to keep responses focused
        temperature=0.1,         # Dropped temperature to minimize creative rambling
        do_sample=False,         # Deterministic greedy decoding
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 5. Clean-up String Parsing: Extract only the line immediately following "Answer:"
raw_answer = generated_text.split("Answer:")[-1].strip()
clean_answer = raw_answer.split("\n")[0].strip()

# 6. Deliverables Display
print("=" * 80)
print(f"{'CORRECTED INTEGRATED RAG PIPELINE OUTPUT':^80}")
print("=" * 80)
print(f"User Question:\n   {user_question}\n")
print(f"Retrieved Context Provided to LLM:\n   {retrieved_chunks}\n")
print("-" * 80)
print(f"Generated Human-Readable Answer:\n   {clean_answer}")
print("=" * 80)

Loading LLM: microsoft/phi-2...


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM loaded successfully!

Generating Answer...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


                    CORRECTED INTEGRATED RAG PIPELINE OUTPUT                    
User Question:
   How many casual leaves are available?

Retrieved Context Provided to LLM:
   Standard Leave Policy Guidelines. Employees are granted a total of 12 casual leaves annually, which accrue on a monthly basis. Additionally, 15 sick leaves are allocated per calendar year for medical emergencies. Maternity leave extends up to 26 weeks of paid time off.

--------------------------------------------------------------------------------
Generated Human-Readable Answer:
   12 casual leaves are available.


In [22]:
import numpy as np
import torch

def ask_policy_assistant(user_question, embedding_model, faiss_index, chunks_list, llm_model, llm_tokenizer, k=3):
    """
    Executes the complete Retrieval-Augmented Generation pipeline.

    Workflow:
    1. Encode the incoming employee question into a vector.
    2. Search the FAISS index for the top-k structurally relevant policy chunks.
    3. Construct a bounded prompt containing context headers.
    4. Pass the prompt to the LLM and prune extra generation loops.
    """
    # Step 1: Generate Embedding for the incoming Question
    query_vector = embedding_model.encode([user_question])
    query_vector_np = np.array(query_vector, dtype=np.float32)

    # Step 2: Search FAISS Index for Top-K Nearest Neighbor Chunks
    distances, indices = faiss_index.search(query_vector_np, k)

    # Compile the retrieved text fragments into a single cohesive context block
    retrieved_contexts = []
    for rank in range(k):
        chunk_idx = indices[0][rank]
        if chunk_idx < len(chunks_list):
            retrieved_contexts.append(chunks_list[chunk_idx].strip())

    context_str = "\n\n".join(retrieved_contexts)

    # Step 3: Construct the Targeted HR System Prompt
    prompt = f"""Instruction: You are an automated HR policy assistant. Provide a single, direct, factual answer to the question using only the context given below. Do not generate alternative scenarios or continue typing after your answer.

Context:
{context_str}

Question:
{user_question}

Answer:"""

    # Step 4: Run Inference via the LLM
    inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.1,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    generated_text = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Step 5: Clean-up Post-Processing Parsing
    raw_answer = generated_text.split("Answer:")[-1].strip()
    clean_answer = raw_answer.split("\n")[0].strip()

    return context_str, clean_answer

## Task 5: RAG Pipeline Evaluation

Objective: Evaluate the retrieval accuracy and answer relevance of the RAG system using a set of sample questions and ground truth answers.

Deliverables:
*   Define sample test questions.
*   Define ground truth answers for each question.
*   Display the retrieved context and generated answer for each question.
*   Provide a basic assessment of retrieval and answer quality.

In [23]:
import time

# Define sample test questions and their ground truth answers
test_cases = [
    {
        "question": "How many sick leaves are available annually?",
        "ground_truth_answer": "Employees receive 15 sick leaves annually.",
        "expected_keywords_retrieval": ["15 sick leaves annually"]
    },
    {
        "question": "Can employees permanently work from home?",
        "ground_truth_answer": "Employees may work from home twice per week.",
        "expected_keywords_retrieval": ["work from home twice per week"]
    },
    {
        "question": "When are travel expenses reimbursed?",
        "ground_truth_answer": "Travel expenses incurred for official company business will be reimbursed within 30 days of submitting a valid expense claim.",
        "expected_keywords_retrieval": ["reimbursed within 30 days"]
    },
    {
        "question": "Are any family members covered by medical insurance?",
        "ground_truth_answer": "The policy offers a maximum annual coverage limit of $50,000 per family unit.",
        "expected_keywords_retrieval": ["per family unit", "medical insurance"]
    },
    {
        "question": "How many casual leaves are available?",
        "ground_truth_answer": "Employees are granted a total of 12 casual leaves annually.",
        "expected_keywords_retrieval": ["12 casual leaves annually"]
    }
]

print("=" * 80)
print(f"{'RAG PIPELINE EVALUATION':^80}")
print("=" * 80)

overall_retrieval_accuracy = 0
overall_answer_relevance = 0
total_response_time = 0

for i, tc in enumerate(test_cases):
    user_question = tc["question"]
    ground_truth_answer = tc["ground_truth_answer"]
    expected_keywords_retrieval = tc["expected_keywords_retrieval"]

    start_time = time.time()
    retrieved_contexts, generated_answer = ask_policy_assistant(
        user_question,
        embedding_model,
        faiss_index,
        chunks_to_embed, # Using chunks_to_embed which contains recursive_chunks
        model,
        tokenizer,
        k=3
    )
    end_time = time.time()
    response_time = end_time - start_time
    total_response_time += response_time

    print(f"\n--- Test Case {i + 1} ---")
    print(f"Question: {user_question}")
    print(f"Ground Truth Answer: {ground_truth_answer}")
    print(f"Response Time: {response_time:.4f} seconds")

    print("\nRetrieved Contexts:")
    retrieval_accurate = True
    for keyword in expected_keywords_retrieval:
        if keyword.lower() not in retrieved_contexts.lower():
            retrieval_accurate = False
            break

    for ctx in retrieved_contexts.split("\n\n"):
        print(f"  - {ctx.strip()}")

    print(f"Retrieval Accuracy: {'✅ Accurate' if retrieval_accurate else '❌ Inaccurate'}")
    if retrieval_accurate: overall_retrieval_accuracy += 1

    print(f"\nGenerated Answer: {generated_answer}")

    # Simple check for answer relevance (can be enhanced for semantic similarity)
    answer_relevant = False
    if ground_truth_answer.lower() in generated_answer.lower() or generated_answer.lower() in ground_truth_answer.lower():
        answer_relevant = True
    else:
      # A more lenient check for partial relevance if direct match fails
      for keyword in expected_keywords_retrieval:
        if keyword.lower() in generated_answer.lower():
          answer_relevant = True
          break


    print(f"Answer Relevance: {'✅ Relevant' if answer_relevant else '❌ Irrelevant'}")
    if answer_relevant: overall_answer_relevance += 1
    print("-" * 80)

print("\n" + "=" * 80)
print(f"{'EVALUATION SUMMARY':^80}")
print("=" * 80)
print(f"Total Test Cases: {len(test_cases)}")
print(f"Retrieval Accuracy Score: {overall_retrieval_accuracy}/{len(test_cases)} ({overall_retrieval_accuracy/len(test_cases):.2%})")
print(f"Answer Relevance Score: {overall_answer_relevance}/{len(test_cases)} ({overall_answer_relevance/len(test_cases):.2%})")
print(f"Average Response Time: {total_response_time / len(test_cases):.4f} seconds")
print("=" * 80)

                            RAG PIPELINE EVALUATION                             

--- Test Case 1 ---
Question: How many sick leaves are available annually?
Ground Truth Answer: Employees receive 15 sick leaves annually.
Response Time: 0.8795 seconds

Retrieved Contexts:
  - must be submitted through the HR portal and approved by the immediate manager at least 5
business days in advance.Standard Leave Policy Guidelines. Employees are granted a total
of 12 casual leaves annually, which accrue on a monthly basis. Additionally, 15 sick leaves
are allocated per calendar year for medical emergencies. Maternity leave extends up to 26
weeks of paid time off, while paternity leave offers 2 weeks of paid time off. All planned leaves
  - of 12 casual leaves annually, which accrue on a monthly basis. Additionally, 15 sick leaves
are allocated per calendar year for medical emergencies. Maternity leave extends up to 26
weeks of paid time off, while paternity leave offers 2 weeks of paid time off. A